# 06 | Bagging and restart ensembles

This experiment asks whether combining rankings from several fitted models improves retrieval over one full-training model. The task is the same six-supergroup retrieval setup used in the preceding experiment: all 20 Newsgroups categories are mapped into six supergroups, with one fixed query per group.

We compare full-data references with five-member ensembles for PLSA, TF-IDF/VSM, and LSI. Bootstrap members use five same-size samples drawn with replacement from the training partition; each saved sample is reused across all three method families. Rankings are combined by mean rank because model scores use different scales. PLSA also has a five-restart ensemble trained on the full training data, which separates gains from random initialization from gains involving bootstrap resampling.


## 1. Settings and imports

The shared preprocessed corpus supplies the documents and original category labels. The experiment-specific supergroup mapping, split, bootstrap samples, queries, and results folder are defined here.


In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.ensemble import average_rank_score
from src.evaluation import evaluate_ranked_retrieval
from src.lsi import fit_lsi
from src.plsa_model import train_plsa_restarts
from src.retrieval import cosine_scores, plsa_query_log_likelihood
from src.text_processing import load_nlp, preprocess_texts
from src.tfidf import fit_tfidf

SEED = 42
TEST_SIZE = 0.20
K = 6
N_RESTARTS = 5
BAGGING_ESTIMATORS = 5
MAX_ITER = 100
TOLERANCE = 1e-4
MAX_FEATURES = 3000
MIN_DF = 3
MAX_DF = 0.90
N_LSI_COMPONENTS = 50
RETRIEVAL_K = 10
FOLD_IN_ITERATIONS = 100
DATA_PATH = REPOSITORY_ROOT / 'data' / '20news_preprocessed.npz'
OUTPUT_DIR = REPOSITORY_ROOT / 'results' / 'bagging_experiment'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPERGROUPS = {
    'Computing': ['comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x'],
    'Digital technology and engineering': ['sci.crypt', 'sci.electronics'],
    'Health and space science': ['sci.med', 'sci.space'],
    'Recreation and consumer exchange': ['misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey'],
    'Religion and worldview': ['alt.atheism', 'soc.religion.christian', 'talk.religion.misc'],
    'Politics and international affairs': ['talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc'],
}
print(f'Repository: {REPOSITORY_ROOT.resolve()}')
print(f'Results folder: {OUTPUT_DIR.resolve()}')
print(f'Ensemble size: {BAGGING_ESTIMATORS}; PLSA K: {K}')


## 2. Load all 20 categories and map them into six supergroups

Every source category belongs to exactly one supergroup. The mapped labels define stratification and retrieval relevance only; PLSA training remains unsupervised. Retaining the full corpus and this mapping keeps the ensemble comparison aligned with the preceding six-supergroup experiment.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run src/preprocessing.ipynb first.')
with np.load(DATA_PATH, allow_pickle=False) as corpus:
    documents = corpus['documents'].tolist()
    original_labels = corpus['labels'].astype(np.int64, copy=False)
    source_category_names = corpus['category_names'].tolist()

mapped_categories = [name for members in SUPERGROUPS.values() for name in members]
missing = sorted(set(source_category_names) - set(mapped_categories))
unknown = sorted(set(mapped_categories) - set(source_category_names))
duplicates = sorted({name for name in mapped_categories if mapped_categories.count(name) > 1})
if missing or unknown or duplicates:
    raise ValueError(f'Supergroups must cover each source category once. Missing={missing}; unknown={unknown}; duplicates={duplicates}')
source_to_supergroup = {
    category: group for group, members in SUPERGROUPS.items() for category in members
}
target_names = list(SUPERGROUPS)
supergroup_to_id = {name: i for i, name in enumerate(target_names)}
labels = np.asarray([
    supergroup_to_id[source_to_supergroup[source_category_names[label]]]
    for label in original_labels
], dtype=np.int64)

category_counts = pd.DataFrame([
    {'supergroup': group, 'source_category': category,
     'documents': int(np.count_nonzero(original_labels == source_category_names.index(category)))}
    for group, members in SUPERGROUPS.items() for category in members
])
group_counts = pd.DataFrame({
    'supergroup': target_names,
    'documents': np.bincount(labels, minlength=len(target_names)),
})
print(f'Documents: {len(documents):,}; original categories: {len(source_category_names)}')
display(category_counts)
display(group_counts)


## 3. Make the shared train/test split and count representation

A stratified 80/20 split reserves the same held-out task for every reference and ensemble. The count vocabulary is fitted on training documents only. Documents emptied by preprocessing are excluded from every ranking so the methods compare the same test candidates.


In [ ]:
docs_train, docs_test, y_train, y_test, original_y_train, original_y_test = train_test_split(
    documents, labels, original_labels,
    test_size=TEST_SIZE, random_state=SEED, stratify=labels,
)
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer(
    lowercase=False, tokenizer=str.split, token_pattern=None,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
    dtype=np.float64,
)
X_train = count_vectorizer.fit_transform(docs_train)
X_test = count_vectorizer.transform(docs_test)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test = np.asarray([bool(text.strip()) for text in docs_test])

split_summary = pd.DataFrame({
    'split': ['train', 'test'],
    'documents': [len(docs_train), len(docs_test)],
    'empty_documents': [sum(not text.strip() for text in docs_train), int((~nonempty_test).sum())],
})
print('Training documents per supergroup:', dict(zip(target_names, np.bincount(y_train, minlength=len(target_names)))))
display(split_summary)
print('Count matrix shapes:', X_train.shape, X_test.shape)


## 4. Fit full-data reference models and the PLSA restart control

The full-data references are trained on every row in the training partition. PLSA is fitted from five random initializations; the highest-training-likelihood fit is the single-model reference, while all five fits are retained for the restart ensemble. The restart ensemble combines initializations without resampling documents.


In [ ]:
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=N_RESTARTS,
    max_iter=MAX_ITER, tol=TOLERANCE, base_seed=SEED,
)
theta_test_full = best_plsa.fold_in(X_test, max_iter=FOLD_IN_ITERATIONS)
restart_theta_test = [
    model.fold_in(X_test, max_iter=FOLD_IN_ITERATIONS)
    for model, _ in restart_runs
]
restart_summary = pd.DataFrame([
    {'restart': i + 1, 'seed': SEED + i, 'iterations': model.n_iter_,
     'training_log_likelihood': float(log_likelihood),
     'selected_as_single_reference': model is best_plsa}
    for i, (model, log_likelihood) in enumerate(restart_runs)
])

tfidf_vectorizer, (X_train_tfidf, X_test_tfidf) = fit_tfidf(
    docs_train, docs_test, max_features=MAX_FEATURES,
    min_df=MIN_DF, max_df=MAX_DF,
)
lsi_model, (lsi_train, lsi_test) = fit_lsi(
    X_train_tfidf, X_test_tfidf,
    n_components=N_LSI_COMPONENTS, seed=SEED,
)
print('PLSA test mixtures:', theta_test_full.shape)
print('TF-IDF shapes:', X_train_tfidf.shape, X_test_tfidf.shape)
print('LSI shapes:', lsi_train.shape, lsi_test.shape)
display(restart_summary)


## 5. Draw shared bootstrap samples and fit PLSA members

Each member draws as many row indices as there are training documents, with replacement. Some training documents appear multiple times and some are omitted. We save these exact index arrays and reuse them for PLSA, TF-IDF, and LSI, pairing the resamples across model families.


In [ ]:
bootstrap_rng = np.random.default_rng(SEED + 500)
bootstrap_indices_list = [
    bootstrap_rng.integers(0, X_train.shape[0], size=X_train.shape[0])
    for _ in range(BAGGING_ESTIMATORS)
]
bootstrap_sampling_summary = pd.DataFrame([
    {'member': member + 1, 'sampled_rows': len(indices),
     'unique_training_documents': len(np.unique(indices)),
     'omitted_training_documents': X_train.shape[0] - len(np.unique(indices))}
    for member, indices in enumerate(bootstrap_indices_list)
])

bagged_plsa_models = []
bagged_theta_test = []
bagged_plsa_summary = []
for member, indices in enumerate(bootstrap_indices_list):
    bootstrap_X = X_train[indices]
    model = train_plsa_restarts(
        bootstrap_X, n_topics=K, n_restarts=1,
        max_iter=MAX_ITER, tol=TOLERANCE,
        base_seed=SEED + 1000 + member,
    )[0]
    bagged_plsa_models.append(model)
    bagged_theta_test.append(model.fold_in(X_test, max_iter=FOLD_IN_ITERATIONS))
    bagged_plsa_summary.append({
        'member': member + 1, 'unique_training_documents': len(np.unique(indices)),
        'iterations': model.n_iter_,
        'bootstrap_training_log_likelihood': float(model.log_likelihoods[-1]),
    })
bagged_plsa_summary = pd.DataFrame(bagged_plsa_summary)
print('Shared bootstrap sample summary:')
display(bootstrap_sampling_summary)
print('Bootstrap PLSA members:')
display(bagged_plsa_summary)


## 6. Fit bootstrap TF-IDF and LSI members

For each shared resample, TF-IDF vocabulary and inverse document frequencies are fitted anew, then LSI is fitted in that member's TF-IDF space. Each member transforms the same held-out documents. The member-specific spaces can differ, so we aggregate their rankings rather than averaging their raw scores or feature vectors.


In [ ]:
bagged_tfidf_vectorizers = []
bagged_tfidf_test = []
bagged_lsi_models = []
bagged_lsi_test = []
baseline_bootstrap_rows = []
for member, indices in enumerate(bootstrap_indices_list):
    bootstrap_docs = [docs_train[i] for i in indices]
    member_vectorizer, (member_train_tfidf, member_test_tfidf) = fit_tfidf(
        bootstrap_docs, docs_test,
        max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
    )
    member_lsi, (member_train_lsi, member_test_lsi) = fit_lsi(
        member_train_tfidf, member_test_tfidf,
        n_components=N_LSI_COMPONENTS, seed=SEED + 1200 + member,
    )
    bagged_tfidf_vectorizers.append(member_vectorizer)
    bagged_tfidf_test.append(member_test_tfidf)
    bagged_lsi_models.append(member_lsi)
    bagged_lsi_test.append(member_test_lsi)
    baseline_bootstrap_rows.append({
        'member': member + 1,
        'bootstrap_rows': len(indices),
        'tfidf_features': member_train_tfidf.shape[1],
        'lsi_dimensions': member_train_lsi.shape[1],
    })
baseline_bootstrap_summary = pd.DataFrame(baseline_bootstrap_rows)
print('Bootstrap TF-IDF/LSI member shapes:')
display(baseline_bootstrap_summary)


## 7. Compare retrieval rankings

Each supergroup has one fixed query. All ten methods rank the same non-empty test documents:

- Full-data TF-IDF and LSI references.
- Five-member TF-IDF and LSI bootstrap ensembles.
- The best full-data PLSA restart, a five-restart full-data PLSA ensemble, and a five-member PLSA bootstrap ensemble, each scored with cosine and query log-likelihood.

For each bootstrap ensemble, every member's scores are converted to zero-based ranks and the negative mean rank is used as the aggregate score. A more relevant document near the top receives a larger aggregate score. P@10, Recall@10, AP@10, and NDCG@10 are reported per query; MAP@10 is mean AP@10 across the six queries.


In [ ]:
supergroup_queries = {
    'Computing': 'computer graphics software hardware operating systems windows',
    'Digital technology and engineering': 'electronics circuits encryption cryptography hardware',
    'Health and space science': 'medicine health disease doctor space NASA planet',
    'Recreation and consumer exchange': 'cars motorcycles baseball hockey sports sale price',
    'Religion and worldview': 'religion atheist christian belief faith god',
    'Politics and international affairs': 'politics government guns international middle east policy',
}
if set(supergroup_queries) != set(target_names):
    raise ValueError('Provide exactly one query for each supergroup.')
query_nlp = load_nlp()
y_test_eval = y_test[nonempty_test]
theta_full_eval = theta_test_full[nonempty_test]
restart_theta_eval = [theta[nonempty_test] for theta in restart_theta_test]
bagged_theta_eval = [theta[nonempty_test] for theta in bagged_theta_test]

retrieval_rows = []
for group_name, query_text in supergroup_queries.items():
    target_label = target_names.index(group_name)
    processed_query = preprocess_texts([query_text], nlp=query_nlp)[0]
    query_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    query_tfidf = tfidf_vectorizer.transform([processed_query])
    query_lsi = lsi_model.transform(query_tfidf)

    full_plsa_theta = best_plsa.fold_in(query_counts, max_iter=FOLD_IN_ITERATIONS)
    full_scores = {
        'TF-IDF/VSM (full)': cosine_scores(query_tfidf, X_test_tfidf[nonempty_test]),
        'LSI (full)': cosine_scores(query_lsi, lsi_test[nonempty_test]),
        'PLSA cosine (best restart)': cosine_scores(full_plsa_theta, theta_full_eval),
        'PLSA query likelihood (best restart)': plsa_query_log_likelihood(
            query_counts, theta_full_eval, best_plsa.phi_, smoothing=1e-10,
        ),
    }

    restart_cosine_scores = []
    restart_likelihood_scores = []
    for (model, _), member_theta in zip(restart_runs, restart_theta_eval):
        query_theta = model.fold_in(query_counts, max_iter=FOLD_IN_ITERATIONS)
        restart_cosine_scores.append(cosine_scores(query_theta, member_theta))
        restart_likelihood_scores.append(plsa_query_log_likelihood(
            query_counts, member_theta, model.phi_, smoothing=1e-10,
        ))
    full_scores['PLSA cosine (5 restarts)'] = average_rank_score(restart_cosine_scores)
    full_scores['PLSA query likelihood (5 restarts)'] = average_rank_score(restart_likelihood_scores)

    bootstrap_plsa_cosine = []
    bootstrap_plsa_likelihood = []
    for model, member_theta in zip(bagged_plsa_models, bagged_theta_eval):
        query_theta = model.fold_in(query_counts, max_iter=FOLD_IN_ITERATIONS)
        bootstrap_plsa_cosine.append(cosine_scores(query_theta, member_theta))
        bootstrap_plsa_likelihood.append(plsa_query_log_likelihood(
            query_counts, member_theta, model.phi_, smoothing=1e-10,
        ))
    full_scores['PLSA cosine (bagged)'] = average_rank_score(bootstrap_plsa_cosine)
    full_scores['PLSA query likelihood (bagged)'] = average_rank_score(bootstrap_plsa_likelihood)

    bootstrap_tfidf_scores = []
    bootstrap_lsi_scores = []
    for vectorizer, member_test_tfidf, member_lsi, member_test_lsi in zip(
        bagged_tfidf_vectorizers, bagged_tfidf_test,
        bagged_lsi_models, bagged_lsi_test,
    ):
        member_query_tfidf = vectorizer.transform([processed_query])
        bootstrap_tfidf_scores.append(cosine_scores(
            member_query_tfidf, member_test_tfidf[nonempty_test],
        ))
        member_query_lsi = member_lsi.transform(member_query_tfidf)
        bootstrap_lsi_scores.append(cosine_scores(
            member_query_lsi, member_test_lsi[nonempty_test],
        ))
    full_scores['TF-IDF/VSM (bagged)'] = average_rank_score(bootstrap_tfidf_scores)
    full_scores['LSI (bagged)'] = average_rank_score(bootstrap_lsi_scores)

    for method, scores in full_scores.items():
        metrics = evaluate_ranked_retrieval(
            scores, y_test_eval, target_label, k=RETRIEVAL_K,
        )
        retrieval_rows.append({
            'query_supergroup': group_name, 'query_text': query_text,
            'method': method,
            'Precision@10': metrics['Precision@k'],
            'Recall@10': metrics['Recall@k'],
            'AP@10': metrics['AP@k'], 'NDCG@10': metrics['NDCG@k'],
            'relevant_test_documents': int(np.count_nonzero(y_test_eval == target_label)),
        })

retrieval_metrics = pd.DataFrame(retrieval_rows)
metric_columns = ['Precision@10', 'Recall@10', 'AP@10', 'NDCG@10']
mean_metrics = (
    retrieval_metrics.groupby('method', sort=False)[metric_columns]
    .mean().rename(columns={'AP@10': 'MAP@10'}).reset_index()
    .rename(columns={'method': 'Method'})
)
print('Per-query retrieval results:')
display(retrieval_metrics.round(4))
print('Macro-average across the six supergroup queries:')
display(mean_metrics.round(4))


## 8. Visualize the ensemble comparison

The charts show one metric per panel so the macro-average comparison is easy to read. These are computed from the six query-level results; no result interpretation is written in advance of running the notebook.


In [ ]:
from textwrap import fill

plot_metrics = ['Precision@10', 'MAP@10', 'NDCG@10']
method_names = mean_metrics['Method'].tolist()
wrapped_method_names = [fill(name, width=22) for name in method_names]
method_colors = dict(zip(method_names, plt.cm.tab10.colors))
retrieval_fig, axes = plt.subplots(
    len(plot_metrics), 1, figsize=(14, 15), constrained_layout=True,
)
for ax, metric in zip(axes, plot_metrics):
    values = mean_metrics[metric].to_numpy(dtype=float)
    bars = ax.bar(
        np.arange(len(values)), values,
        color=[method_colors[name] for name in method_names],
    )
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xticks(np.arange(len(values)))
    ax.set_xticklabels(wrapped_method_names, rotation=0, ha='center', fontsize=8)
    ax.set_title(metric)
    ax.set_ylabel('Macro-average')
    ax.set_ylim(0, min(1.15, max(0.1, float(values.max()) * 1.2)))
    ax.grid(axis='y', alpha=0.3)
retrieval_fig.suptitle('Full-data and ensemble retrieval performance')


## 9. Save outputs

Per-query and macro-average metrics, restart and bootstrap summaries, the exact shared bootstrap indices, the figure, and run settings are saved in `results/bagging_experiment/`. Rerunning this notebook replaces the previous outputs for this experiment.


In [ ]:
retrieval_metrics.to_csv(OUTPUT_DIR / 'retrieval_metrics_by_query.csv', index=False)
mean_metrics.to_csv(OUTPUT_DIR / 'retrieval_metrics_macro_average.csv', index=False)
restart_summary.to_csv(OUTPUT_DIR / 'plsa_full_data_restart_summary.csv', index=False)
bagged_plsa_summary.to_csv(OUTPUT_DIR / 'plsa_bootstrap_member_summary.csv', index=False)
baseline_bootstrap_summary.to_csv(OUTPUT_DIR / 'baseline_bootstrap_member_summary.csv', index=False)
bootstrap_sampling_summary.to_csv(OUTPUT_DIR / 'bootstrap_sampling_summary.csv', index=False)
np.savez_compressed(
    OUTPUT_DIR / 'bootstrap_indices.npz',
    **{f'member_{i + 1}': indices for i, indices in enumerate(bootstrap_indices_list)},
)
retrieval_fig.savefig(OUTPUT_DIR / 'retrieval_ensemble_comparison.png', dpi=160, bbox_inches='tight')

configuration = {
    'experiment': 'Bagging and restart ensembles',
    'seed': SEED, 'test_size': TEST_SIZE, 'n_topics': K,
    'full_data_plsa_restarts': N_RESTARTS,
    'bootstrap_estimators': BAGGING_ESTIMATORS,
    'bootstrap_sample_size': int(X_train.shape[0]),
    'bootstrap_seed': SEED + 500,
    'bootstrap_with_replacement': True,
    'bootstrap_resamples_shared_across': ['PLSA', 'TF-IDF/VSM', 'LSI'],
    'aggregation': 'negative mean of zero-based member ranks; higher is better',
    'max_iter': MAX_ITER, 'tolerance': TOLERANCE,
    'max_features': MAX_FEATURES, 'min_df': MIN_DF, 'max_df': MAX_DF,
    'lsi_components_requested': N_LSI_COMPONENTS,
    'retrieval_k': RETRIEVAL_K,
    'supergroups': SUPERGROUPS, 'queries': supergroup_queries,
    'documents': len(documents), 'training_documents': len(docs_train),
    'test_documents': len(docs_test),
}
with (OUTPUT_DIR / 'configuration.json').open('w', encoding='utf-8') as handle:
    json.dump(configuration, handle, indent=2)
print(f'Saved bagging experiment outputs to {OUTPUT_DIR.resolve()}')
